In [1]:
import sys, pygame, math, random, pygame.gfxdraw, pygame.transform
import numpy as np

pygame 2.5.2 (SDL 2.28.3, Python 3.9.13)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
def get_distance(x1,x2,y1,y2):
    distance = -1
    dx = x2 - x1
    dy = y2 - y1
    distance = pow(dx,2)+pow(dy,2)
    distance = math.sqrt(distance)
    return distance

In [3]:
class Ball:
    image = pygame.image.load("seed_primordium.png")
    bouncemod = 0.9
    size = 1
    def __init__(self,x,y,dx,dy):
        self.rect = self.image.get_rect()
        self.x = x
        self.y = y
        self.rect = self.rect.move([x,y])
        self.dx = dx
        self.dy = dy
        
    def __str__(self):
        return f"({self.rect.x},{self.rect.y})"
    
    def clone(self):
        newball = Ball(self.rect.x,self.rect.y,self.dx,self.dy)
        newball.size = self.size
        return newball
    
    def get_x(self):
        return self.x
    
    def get_y(self):
        return self.y
    
    def set_x(self,x):
        self.x = x
        self.rect.x = x
        
    def set_y(self,y):
        self.y = y
        self.rect.y = self.y

    def move(self):
        self.x += self.dx
        self.rect.x = self.x
        self.y += self.dy
        self.rect.y = self.y
        
    def collide(self,xmin,xmax,ymin,ymax):
        if self.get_x() <= xmin:
            self.set_x(0)
            self.dx = abs(self.dx*self.bouncemod)
        if self.get_x() >= xmax-self.rect.width:
            self.set_x(xmax-self.rect.width)
            self.dx = -abs(self.dx*self.bouncemod)
        if self.get_y() <= ymin:
            self.set_y(0)
            self.dy = abs(self.dy*self.bouncemod)
        if self.get_y() >= ymax-self.rect.height:
            self.set_y(ymax-self.rect.height)
            self.dy = -abs(self.dy*self.bouncemod)
            
    def resize(self,factor): #not perfect CHANGE to use internal values.
        self.size = self.size * factor
        self.image = pygame.transform.scale_by(self.image,self.size) 
        self.rect = self.rect.scale_by(self.size,self.size) 
    
    def vector(self):
        return [self.dx,self.dy]

In [4]:
def two_balls_touching(b1,b2):
    #if the two balls are touching
    radius1 = (b1.rect.size[1] * b1.size)/2
    radius2 = (b2.rect.size[1] * b2.size)/2
    distance = get_distance(b1.get_x(),b2.get_x(),b1.get_y(),b2.get_y())
    if distance <= (radius1 + radius2):#the distance between them is less than both their radii
        return True
    else: return False

In [5]:
def optimal_ball_location(origin, radius, nodes):
    degrees = 360
    point = [10,10] #dummy
    
    xpoints = []
    ypoints = []
    for i in range(degrees):
        xpoints.append(i)
        
    for degree in xpoints:
        #print(degree)
        #find the point on the circle from which the distance will be calculated
        x = (math.cos(math.radians(degree))*radius) + origin[0]
        y = (math.sin(math.radians(degree))*radius) + origin[1]
        point = [x,y]
        #find the sum of distance from all 'nodes' to this point
        distance = 0
        for node in nodes:
            partialdistance = get_distance(x,node.get_x(),y,node.get_y())
            if partialdistance == 0:
                partialdistance = 0.0000001
            distance += 1/partialdistance
        #print(point)
        #print(distance)
        ypoints.append(distance)
    
    #choose a random degree value (0-360)
    pointerx = random.randint(1,degrees)
    margin = 70 #to start
    while margin > 0:
        # find the slope from neighboring points (start with a margin of 25 each side)
        leftx = int(pointerx-margin)
        if leftx < 0:
            leftx += degrees
        rightx = int(pointerx+margin)
        if rightx >= 360:
            rightx -= degrees
        slope = ((ypoints[rightx]-ypoints[leftx])/(margin*2))#rise/run
        #print(pointerx)
        #print(slope)
        # move in the direction of neighboring slope (same margin)
        if slope >= 0:
            pointerx -= margin
        else:
            pointerx += margin
        if pointerx >= degrees:
            pointerx -= degrees
        if pointerx < 0:
            pointerx += degrees
        # do this until the margin is 0
        margin -= 5
    #print('radius',radius)
    print(pointerx)
    x = (math.cos(math.radians(pointerx))*radius) + origin[0]
    y = (math.sin(math.radians(pointerx))*radius) + origin[1]
    point = [x,y]
    return point

In [6]:
def prescribe_ball_location(origin,radius,previous):
    point = [10,10] #dummy
    newangle = previous + 138
    if newangle > 360:
        newangle -= 360
    x = (math.cos(math.radians(newangle))*radius) + origin[0]
    y = (math.sin(math.radians(newangle))*radius) + origin[1]
    point = [x,y]
    print(newangle)
    return [point,newangle]

In [7]:
def exp_decay(dx,dy):
    return (dx,dy)

In [8]:
pygame.init()

clock = pygame.time.Clock()
size = width, height = 700, 700
black = 0, 0, 0

screen = pygame.display.set_mode(size)

speedmod = 0.998 #998 is a good one

ball_list = []
#TEMP: create 3 balls in vicinity. 
#while len(ball_list) < 3:
    #x = random.randint(0,700)
    #y = random.randint(0,700)
    #if (x <= (300-60) or x >= (300+60)) and (y <= (300-60) or y >= (300+60)):
        #newball = Ball(x,y,0,0)
        #print(x,y)
        #ball_list.append(newball)

#create the player's ball.
playerball = Ball(300,300,0,0)
playerball.image = pygame.image.load("Meristem_green.png")
playerball.cooldown = 100

previousangle = 0

while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT: 
            pygame.display.quit()
            sys.exit("Exit.")
    
    #move the balls
    for ball in ball_list:
        ball.move()
        #ball.collide(0,700,0,700)
        #TODO TODO: they should not slow down at first, then slow down a lot.
        #exponential?
        if False:
            ball.dx = ball.dx * speedmod
            ball.dy = ball.dy * speedmod
        if True: 
            dx = ball.dx
            dy = ball.dy
            vector = exp_decay(dx,dy)
            ball.dx = vector[0]
            ball.dy = vector[1]
        #ball.dx = pow(ball.dx,speedmod)
        #ball.dy = pow(ball.dy,speedmod)
        
    #cooldown the meristem (playerball where new balls will shoot off)
    if playerball.cooldown > 0:
        playerball.cooldown -= 1
        
    #affect the player's movement: NOT NECESSARY
    keys = pygame.key.get_pressed()
    if keys[pygame.K_w]:
        playerball.dy -= speedmod * dt
    if keys[pygame.K_s]:
        playerball.dy += speedmod * dt
    if keys[pygame.K_a]:
        playerball.dx -= speedmod * dt
    if keys[pygame.K_d]:
        playerball.dx += speedmod * dt
        
    #QUIT THE GAME.
    if keys[pygame.K_ESCAPE]:
        pygame.display.quit()
        sys.exit("Escape!")
    if keys[pygame.K_q]:
        sys.exit("Quit.")
    
    #TODO: place a new ball away from balls in a certain radius...
    #?combine vectors, normalize, and reverse?? 
    #or do a gradient descent algorithm...
    if playerball.cooldown <= 1:
        playerball.cooldown = 20 #reset cooldown
        newball = Ball(playerball.get_x(),playerball.get_y(),0,0)
        
        #find the list of local balls to the playerball
        local_dist = 200
        local_balls = []
        for ball in ball_list:
            distance = get_distance(playerball.get_x(),ball.get_x(),playerball.get_y(),ball.get_y())
            if distance < local_dist:
                local_balls.append(ball)
        #print(len(local_balls))
        
        #find an optimal location for the new ball
        playerballradius = playerball.rect.size[0]/2
        #newballxy = optimal_ball_location([playerball.get_x()+playerballradius/2,playerball.get_y()+playerballradius/2],playerballradius,local_balls)
        answer = prescribe_ball_location([playerball.get_x()+playerballradius/2,playerball.get_y()+playerballradius/2],playerballradius,previousangle)
        newballxy = answer[0]
        previousangle = answer[1]
        newball.set_x(newballxy[0])
        newball.set_y(newballxy[1])
        #TODO: set the newball moving directly away from playerball
        #find the slope between the center of the playerball and the center of the newball
        playerballcenter = [playerball.get_x()+playerballradius,playerball.get_y()+playerballradius]
        newball.resize(0.5)
        newballradius = newball.rect.size[0]/2
        newballcenter = [newball.get_x()+newballradius,newball.get_y()+newballradius]
        ydev = playerballcenter[1]-newballcenter[1]
        xdev = playerballcenter[0]-newballcenter[0]
        #set the newball moving along a product of that slope
        newball.dx = -xdev *0.006
        newball.dy = -ydev *0.006
        #debug: find the speed of the balls. varies between 0.5515 and 0.5583 varying by 1%!
        #magnitude = get_distance(0,newball.dx,0,newball.dy)
        #print(magnitude)
        #TODO: find the angle of the ball coming out and find the deviation from last ball.
        
        ball_list.append(newball)
    
    screen.fill(black)
    #blit the balls on screen
    screen.blit(playerball.image,playerball.rect)
    for ball in ball_list:   
        screen.blit(ball.image, ball.rect)
    pygame.display.flip()

    # limits FPS to 60
    # dt is delta time in seconds since last frame, used for framerate-
    # independent physics.
    dt = clock.tick(60) / 1000

138
276
54
192
330
108
246
24
162


SystemExit: Exit.

/Users/adrianchapman/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3465: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
